# 计算 Gini 系数

> 对应代码：`MathStatsCode/code_in_notes/Appendix.B/ginindex.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：编写 ginindex 程序：按收入排序计算累积收入比例与洛伦兹曲线面积，得到 Gini 系数，并支持 by 分组。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Appendix.B`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Appendix.B")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 用程序 `ginindex` 计算**Gini 系数**，完整展示了基于洛伦兹曲线面积的定义式。程序逻辑：
- `program define ginindex, byable(onecall)`：`byable(onecall)` 使命令支持 `by` 前缀（如按省分组计算）；`syntax varlist(max=1), gen(name)` 要求一个收入变量并以 `gen()` 指定新变量的名字；
- 兼容 `by` 前缀：若用户没有 `bysort`，程序自动生成一个恒为1的 `class` 变量当作唯一分组（`_byvars` 由 byable 机制自动维护）；
- `sort byvars 收入` 后按组内排名 `rank`；`egen count/total, by()` 得到组内人数与总收入；`cum` 为累积收入（用 `[n-1]` 逐行累加）；`rat=cum/total` 是累积收入比例；
- **面积计算**：把洛伦兹曲线下的面积近似成一组柱形——第 rank 个个体贡献矩形面积 `rat/num`；再用 `delta` 减去柱子顶部的小三角形（对第一根柱子减 rat/2，其余减相邻高度差的一半/分份数），使面积估计更精确（梯形法思想）；
- `B` 为洛伦兹曲线下面积，Gini = 1-2B（正方形面积1，洛伦兹曲线与对角线之间的面积为 1/2-Gini/2）。

最后两行是使用示例：`ginindex fincome1, gen(gini_all)` 计算全体样本的 Gini；`bysort provcd14: ginindex fincome1, gen(gini)` 按省分组计算（注意 by 前缀下新变量在组内生成、组间拼回同一列）。


In [ ]:
%%stata
cap program drop ginindex
program define ginindex, byable(onecall)
  version 15
  syntax varlist(max=1), gen(name)
quietly{
  if "`_byvars'"==""{
    tempvar class
    gen `class'=1
    local _byvars "`class'"
  }
  // 临时变量
  tempvar rank totalincome cum rat num rect B delta
  // 排序并产生排序变量
  sort `_byvars' `varlist'
  by `_byvars': gen `rank'=_n
  // 产生地区户数、总收入和累积收入及比例
  egen `num'=count(`varlist'), by(`_byvars')
  egen `totalincome'=total(`varlist'), by(`_byvars')
  gen `cum'=`varlist' if `rank'==1
  by `_byvars': replace `cum'=`cum'[_n-1]+`varlist'[_n] if `rank'>1
  gen `rat'=`cum'/`totalincome'
  // 计算每个个体的柱形面积
  by `_byvars': gen `rect'=`rat'/`num'
  // 调整小样本误差，即减去柱子最上面小三角的面积
  gen `delta'=`rat'/2 if `rank'==1
  by `_byvars': replace `delta'=/*
          */  `rat'[_n]-`rat'[_n-1] if `rank'>1
  replace `rect'=`rect'-`delta'/`num'/2
  //最终的指数
  egen `B'=total(`rect'), by(`_byvars')
  gen `gen'=1-2*`B'
}
end

use "../datasets/cfps_family_econ.dta", clear
ginindex fincome1, gen(gini_all)
bysort provcd14: ginindex fincome1, gen(gini)


以上 do 文件中的命令都封装在 `quietly` 内部，运行时不会自动显示结果。下面补充一段展示命令（do 文件中没有，仅用于 notebook 中查看计算结果）：


In [ ]:
%%stata
// 查看计算结果：全体样本的 Gini 系数与各省 Gini 系数的分布
su gini_all gini
